In [ ]:
import pandas as pd
import os
os.environ['USE_PYGEOS'] = '0'
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import geopandas as gpd
import unicodedata
from matplotlib.colors import LogNorm

### Directories
dropbox_dir  =  os.path.join(os.path.expanduser("~"), "Dropbox","Projects")
base_dir     =  os.path.join(dropbox_dir,  "Crop_misallocation")
crop_sub_dir =  os.path.join(dropbox_dir,  "Maize_prediction")
poppy_dir    =  os.path.join(dropbox_dir,  "Maize_prediction")
data_dir     =  os.path.join(base_dir,     "Data/")   # capital D on this machine
plot_dir     =  os.path.join(poppy_dir,     "plots/")
inegi_dir    =  os.path.join(data_dir,     "INEGI/")
siap_dir     =  os.path.join(crop_sub_dir, "Data", "SIAP", "Cleaned")
md_lab_dir   =  os.path.join(inegi_dir,    "Microdata_lab_outputs")
adc_07_dir   =  os.path.join(md_lab_dir,   "CA07_ADC_prod_data_2023-05-09")
sciaga_dir   =  os.path.join(data_dir,     "SCIAGA")
drive_dir    =  os.path.join(poppy_dir,    "Intermediates")   # was drive-download-intermediates
agland_dir   =  os.path.join(poppy_dir,    "Data", "SIAP_agland", "Output")
joel_dir     =  os.path.join(poppy_dir,    "Data", "predictions")
amca_dir     =  os.path.join(poppy_dir,    "Data", "INEGI", "Areas_Censal_Agropecuario_2016")
py_md_lab_dir=  os.path.join(poppy_dir,    "Data", "INEGI", "MD_lab_outputs")
adc_16_dir   =  os.path.join(py_md_lab_dir,"AMCA_ADC_data_2022-11-23")
ca2022_dir   =  os.path.join(py_md_lab_dir,"CA22_mun_yield_2024-03-25")
ca2022_adc_d =  os.path.join(py_md_lab_dir,"LM2304-CA22-2025-09-29-superficie_ENTREGA")

### Inputs
map_path        =  os.path.join(crop_sub_dir, "Data", "Shapefiles", "Municipality_shp", "MUNICIPIOS.shp")   # self-contained copy (2026-09-03)
state_path      =  os.path.join(crop_sub_dir, "Data", "Shapefiles", "Municipality_shp", "STATES.shp")
map_adc_path    =  os.path.join(os.path.expanduser("~"), "Dropbox", "Projects", "Maize_prediction", "Data", "Shapefiles", "adc_shapefile.shp")
adc_2016_shp    =  os.path.join(amca_dir,     "census_areas.shp")
siap_data       =  os.path.join(siap_dir,     "siap_ag_prod_estimation_ca2007.dta")

### ADC data from INEGI
adc_07_data      =  os.path.join(adc_07_dir,   "rendimiento_agr_adc.dta")
adc_07_2016      =  os.path.join(py_md_lab_dir,"ca2007_maize_amca_adcs.dta")
adc_16_data      =  os.path.join(adc_16_dir,   "amca_sup_adc.csv")
adc_22_07        =  os.path.join(ca2022_adc_d, "adc_land_use_ca22_adc07.dta")
adc_22_07_szn    =  os.path.join(ca2022_adc_d, "adc_land_szn_ca22_adc07.dta")

### CA2022 data at municipality level
mun_22_data      =  os.path.join(ca2022_dir,   "mun_land_use_ca22.dta")
mun_22_szn       =  os.path.join(ca2022_dir,   "mun_land_szn_ca22.dta")

### ADC agland data
adc_land_area_07 =  os.path.join(agland_dir,"2007_adcs_agland_area.csv")
adc_land_area_16 =  os.path.join(agland_dir,"2016_adcs_agland_area.csv")

### All predictions from Joel
mun_preds        =  os.path.join(drive_dir,    "muni_ls_preds.csv")
adc_preds        =  os.path.join(drive_dir,    "adcs07_yield_pred.csv")
ha_preds         =  os.path.join(drive_dir,    "adc_ha_hats.csv")
### New 2024 predictions
adc_2024         =  os.path.join(joel_dir,     "adcs_yield_preds.csv")
adc_2022_pred    =  os.path.join(joel_dir,     "adcs_yield_preds_2022.csv") ### Just output for only 2022 predictions
mun_2022_pred    =  os.path.join(joel_dir,     "mun2022_yield_preds.csv")   ### Just output for only 2022 predictions
### New 2025 predictions for ADCs
adc_2022_07_pred =  os.path.join(joel_dir,     "adc_alpha_earth_preds.csv") ### Just output for only 2022 predictions using 2007 ADCs
adc_2022_07_pred =  os.path.join(joel_dir,     "adc_mlp_yield_preds.csv") ### AlphaEarth predictions for 2017-2024 using 2007 ADCs

### INEGI mun. yield data for 2007
mun_yield_07     =  os.path.join(py_md_lab_dir,   "mun_avo_yield_2007.csv")

### Outputs
map_mun_path_ca2007     =  os.path.join(sciaga_dir, "CA2007_mun_fromadc_poly.shp")
adc_expost_corr_pred_07 =  os.path.join(joel_dir, "adc_yield_preds_corrected_2007.csv")
adc_expost_corr_pred_22 =  os.path.join(joel_dir, "adc_yield_preds_corrected_2022.csv")

### Programs
def compute_r_squared(dataframe, column1, column2):
    # Check if the columns exist in the DataFrame
    if column1 not in dataframe.columns or column2 not in dataframe.columns:
        # 2026-08-28: stale interactive diagnostics reference columns from
        # inactive weighting schemes; warn instead of crashing the notebook.
        print(f"compute_r_squared: missing column(s) {column1!r}/{column2!r} -- skipped")
        return np.nan

    # Drop rows with NaN or Inf values in the selected columns
    dataframe = dataframe.replace([np.inf, -np.inf], np.nan).dropna(subset=[column1, column2])

    if len(dataframe) < 2:
        print(f"compute_r_squared: <2 valid rows for {column1!r}/{column2!r} -- skipped")
        return np.nan
    # Calculate the correlation coefficient and square it to get R2
    correlation_coefficient, _ = stats.pearsonr(dataframe[column1], dataframe[column2])
    r_squared = correlation_coefficient ** 2

    return r_squared

### Read in municipality level shapefile.
municipio_shapefile            =  gpd.read_file(map_path)
municipio_shapefile['muncode'] =  municipio_shapefile['CVE_ENT']+municipio_shapefile['CVE_MUN']
municipio_shapefile['muncode'] =  municipio_shapefile['muncode'].astype(int).astype(str)
# municipio_shapefile = municipio_shapefile.set_index(municipio_shapefile.muncode.astype(int))
### Generate representative point for plotting
municipio_shapefile['repx']    =  municipio_shapefile['geometry'].representative_point().x
municipio_shapefile['repy']    =  municipio_shapefile['geometry'].representative_point().y
municipio_shapefile['ha_area'] = municipio_shapefile.to_crs(epsg=6372).geometry.area/10000.0

states_shapefile               =  gpd.read_file(state_path)
states_shapefile               =  states_shapefile.set_index(states_shapefile.muncode.astype(int))

st_line_shp = states_shapefile.copy()
st_line_shp['geometry']= st_line_shp['geometry'].boundary



In [ ]:
# ### Compute R2 for CA2022 data at ADC level (ADCs from 07 census)
adc_22_07_df = pd.read_stata(adc_22_07)
adc_22_07_df = adc_22_07_df[adc_22_07_df['name']=='Maize']
adc_22_07_df = adc_22_07_df[['name', 'adc','muncode','share_irrig','land_input','vol_output']+[c for c in adc_22_07_df.columns if 'yield' in c]]
### Read in seasonal data
adc_22_07_szn_df = pd.read_stata(adc_22_07_szn)
adc_22_07_szn_df = adc_22_07_szn_df[adc_22_07_szn_df['name']=='Maize']
adc_22_07_oi_df  = adc_22_07_szn_df[adc_22_07_szn_df['type']=="o-i"].copy()
adc_22_07_oi_df  = adc_22_07_oi_df[['name', 'adc','muncode']+[c for c in adc_22_07_oi_df.columns if 'yield' in c]]
adc_22_07_oi_df.columns = ['name', 'adc', 'muncode']+[c.replace('yield', 'yield_oi') for c in adc_22_07_oi_df.columns if 'yield' in c]
adc_22_07_pv_df  = adc_22_07_szn_df[adc_22_07_szn_df['type']=="p-v"].copy()
adc_22_07_pv_df  = adc_22_07_pv_df[['name', 'adc', 'muncode']+[c for c in adc_22_07_pv_df.columns if 'yield' in c]]
adc_22_07_pv_df.columns = ['name', 'adc', 'muncode']+[c.replace('yield', 'yield_pv') for c in adc_22_07_pv_df.columns if 'yield' in c]
adc_22_07_df = adc_22_07_df.merge(adc_22_07_oi_df, on=['name', 'adc','muncode'], how='outer').merge(adc_22_07_pv_df, on=['name', 'adc','muncode'], how='outer')
### Merge with Joel's predictions for 2022 at ADC 2007 level
adc_22_07_pred_df = pd.read_csv(adc_2022_07_pred)
adc_22_07_pred_df['adcid'] = adc_22_07_pred_df['adcid'].apply(lambda x: str(x).replace('-',''))
adc_22_07_pred_df['muncode'] = adc_22_07_pred_df['adcid'].str.slice(0,5)
mun_pred_df = adc_22_07_pred_df.groupby(['muncode','year'])[['pred_quantity','pred_area']].sum().reset_index()
mun_pred_df['pred_yield'] = mun_pred_df['pred_quantity']/mun_pred_df['pred_area']

adc_22_07_pred_df = adc_22_07_pred_df[adc_22_07_pred_df['year']==2022]
adc_22_07_pred_df = adc_22_07_pred_df.rename(columns={'adcid':'adc','pred_yield':'yield_pred_2022'})
adc_22_07_pred_df = adc_22_07_pred_df.drop(columns=['year','muncode'])
adc_22_07_df = adc_22_07_df.merge(adc_22_07_pred_df, on='adc', how='outer')
### Merge with SIAP
siap_df          =  pd.read_stata(siap_data)
siap_df          =  siap_df[siap_df['name'] == 'Maize']
siap_df['yield'] =  siap_df['q']/siap_df['ha_planted']
siap_df['muncode'] = siap_df['muncode'].apply(lambda x: str(x).zfill(5))
mun_pred_df      = mun_pred_df.merge(siap_df, on=['muncode','year'], how='outer')
siap_df          =  siap_df[siap_df['year'] == 2022]
siap_df          = siap_df[['muncode','lshare','yield']]
siap_df.columns  = ['muncode', 'share_crop', 'yield_siap']
adc_22_07_df     = adc_22_07_df.merge(siap_df, on='muncode', how='left')

In [ ]:
# compute_r_squared(mun_pred_df, 'yield', 'pred_yield') # R2 = 0.56 -- SIAP
# compute_r_squared(mun_pred_df[mun_pred_df['year']==2024], 'yield', 'pred_yield') # R2 = 0.49 -- SIAP
# compute_r_squared(mun_pred_df[mun_pred_df['year']==2023], 'yield', 'pred_yield') # R2 = 0.52 -- SIAP
# compute_r_squared(mun_pred_df[mun_pred_df['year']==2022], 'yield', 'pred_yield') # R2 = 0.56 -- SIAP
# compute_r_squared(mun_pred_df[mun_pred_df['year']==2021], 'yield', 'pred_yield') # R2 = 0.6 -- SIAP
# compute_r_squared(mun_pred_df[mun_pred_df['year']==2020], 'yield', 'pred_yield') # R2 = 0.61 -- SIAP
# compute_r_squared(mun_pred_df[mun_pred_df['year']==2019], 'yield', 'pred_yield') # R2 = 0.6 -- SIAP
# compute_r_squared(mun_pred_df[mun_pred_df['year']==2018], 'yield', 'pred_yield') # R2 = 0.6 -- SIAP
# compute_r_squared(mun_pred_df[mun_pred_df['year']==2017], 'yield', 'pred_yield') # R2 = 0.57 -- SIAP

In [ ]:
### Compute avg. yield based on ADC predictions and different weighting schemes
adc_df = adc_22_07_df.copy()
### And use this to compute ex-post correction
weight_schemes = ['land_input']
for sum_var in weight_schemes:
   adc_df['pred_Q_'+sum_var]    =  adc_df['yield_pred_2022'] * adc_df[sum_var] ### Predicted yield * area
   adc_df['to_sum_'+sum_var]    =  adc_df.apply(lambda x: x[sum_var]     if not np.isnan(x['yield_pred_2022']) else 0, axis = 1)

adc_df['to_sum_Q']              =  adc_df.apply(lambda x: x['vol_output'] if not np.isnan(x['yield_pred_2022']) else 0, axis = 1)
adc_df['num_adcs']              =  adc_df.apply(lambda x: 1 if not np.isnan(x['yield_pred_2022']) else 0, axis = 1)

mun_sum_cols = ['pred_Q_'+wght for wght in weight_schemes]
mun_sum_cols = mun_sum_cols+['vol_output','land_input','num_adcs','to_sum_Q']
mun_sum_cols = mun_sum_cols+['to_sum_'+wght for wght in weight_schemes]
### Add Joel's predicted area + quantity
mun_sum_cols = mun_sum_cols + ['pred_area','pred_quantity']

avg_yield_pred_df = adc_df.groupby('muncode')[mun_sum_cols].sum().reset_index()

for sum_var in weight_schemes:
   avg_yield_pred_df['yield_pred_'+sum_var] = avg_yield_pred_df['pred_Q_'+sum_var]/avg_yield_pred_df['to_sum_'+sum_var]

avg_yield_pred_df['yield_agg_exceptions']   = avg_yield_pred_df['to_sum_Q']/avg_yield_pred_df['to_sum_land_input']
avg_yield_pred_df['yield_real']             = avg_yield_pred_df['vol_output']/avg_yield_pred_df['land_input']
avg_yield_pred_df['yield_joel_upscaled']    = avg_yield_pred_df['pred_quantity']/avg_yield_pred_df['pred_area']

avg_yield_pred_cols = ['muncode','yield_agg_exceptions','yield_real','yield_joel_upscaled','num_adcs']+['yield_pred_'+wght for wght in weight_schemes]
avg_yield_pred_df = avg_yield_pred_df[avg_yield_pred_cols]
avg_yield_pred_df = avg_yield_pred_df.merge(siap_df, on = 'muncode', how = 'left') ### SIAP mun. avg. yield 
# avg_yield_pred_df = avg_yield_pred_df.merge(mun_avg_adc_yield_df, on = 'muncode', how = 'left') ### Treat agg. CA data as SIAP mun. avg. yield.


for sum_var in weight_schemes:
   avg_yield_pred_df['diff_'+sum_var]     = avg_yield_pred_df['yield_pred_'+sum_var] - avg_yield_pred_df['yield_siap']
   ### Using true ADC avg. yield from CA22
   # avg_yield_pred_df['truediff_'+sum_var] = avg_yield_pred_df['yield_pred_'+sum_var] - avg_yield_pred_df['mun_adc_avg_yield']
   avg_yield_pred_df['diff_ratio_'+sum_var]    = avg_yield_pred_df['yield_pred_'+sum_var]/avg_yield_pred_df['yield_siap']
   # avg_yield_pred_df['truediff_ratio_'+sum_var]= avg_yield_pred_df['yield_pred_'+sum_var]/avg_yield_pred_df['mun_adc_avg_yield']

# avg_yield_pred_df['diff_hapred']     = (avg_yield_pred_df['ha_pred'] - avg_yield_pred_df['ha_planted'])/avg_yield_pred_df['num_adcs']
# avg_yield_pred_df['truediff_hapred'] = (avg_yield_pred_df['ha_pred'] - avg_yield_pred_df['sup_sem'])/avg_yield_pred_df['num_adcs']

avg_yield_pred_df['diff_joel_upscaled']     = avg_yield_pred_df['yield_joel_upscaled'] - avg_yield_pred_df['yield_siap']
avg_yield_pred_df['diff_joel_upscaled_ratio']    = avg_yield_pred_df['yield_joel_upscaled']/avg_yield_pred_df['yield_siap']

tosub_df = avg_yield_pred_df[['muncode']+[col for col in avg_yield_pred_df.columns if 'diff_' in col]]
adc_rc_df = adc_df.merge(tosub_df, on = 'muncode', how = 'left')

for sum_var in weight_schemes:
    ### Correct by ADC avg. SIAP yield
    adc_rc_df['corr_yield_pred_'+sum_var] = adc_rc_df['yield_pred_2022']-adc_rc_df['diff_'+sum_var]
    adc_rc_df['corr_yield_pred_'+sum_var] = adc_rc_df['corr_yield_pred_'+sum_var].apply(lambda x: 0 if x < 0 else x)
    adc_rc_df['corr_yield_pred_'+sum_var] = adc_rc_df.apply(lambda x: x['corr_yield_pred_'+sum_var] if not np.isnan(x['yield_pred_2022']) else np.nan, axis = 1)
   
    ### Correct by true ADC avg. yield
   #  adc_rc_df['ca_avg_corr_yield_'+sum_var] = adc_rc_df['yield_pred_2022']-adc_rc_df['truediff_'+sum_var]
   #  adc_rc_df['ca_avg_corr_yield_'+sum_var] = adc_rc_df['ca_avg_corr_yield_'+sum_var].apply(lambda x: 0 if x < 0 else x)
   #  adc_rc_df['ca_avg_corr_yield_'+sum_var] = adc_rc_df.apply(lambda x: x['ca_avg_corr_yield_'+sum_var] if not np.isnan(x['yield_pred_2022']) else np.nan, axis = 1)

    ### Correct by ADC avg. SIAP yield (mult)
    adc_rc_df['corr_yield_scl_pred_'+sum_var] = adc_rc_df['yield_pred_2022']/adc_rc_df['diff_ratio_'+sum_var]
   #  adc_rc_df['ca_avg_corr_yield_scl_pred_'+sum_var] = adc_rc_df['yield_pred_2022']/adc_rc_df['truediff_ratio_'+sum_var]

adc_rc_df['corr_yield_pred_ju'] = adc_rc_df['yield_pred_2022'] - adc_rc_df['diff_joel_upscaled']
adc_rc_df['corr_yield_pred_ju'] = adc_rc_df['corr_yield_pred_ju'].apply(lambda x: 0 if x < 0 else x)
adc_rc_df['corr_yield_pred_ju'] = adc_rc_df.apply(lambda x: x['corr_yield_pred_ju'] if not np.isnan(x['yield_pred_2022']) else np.nan, axis = 1)

adc_rc_df['corr_yield_scal_pred_ju'] = adc_rc_df['yield_pred_2022']/adc_rc_df['diff_joel_upscaled_ratio']

In [ ]:
# compute_r_squared(adc_rc_df, 'yield', 'corr_yield_pred_land_input') # R2 = 0.61 -- 22CA
# compute_r_squared(adc_rc_df, 'yield', 'corr_yield_scl_pred_land_input') # R2 = 0.59 -- 22CA
# compute_r_squared(adc_rc_df, 'yield', 'corr_yield_pred_ju') # R2 = 0.61 -- 22CA
# compute_r_squared(adc_rc_df, 'yield', 'corr_yield_scal_pred_ju') # R2 = 0.61 -- 22CA


In [ ]:
# compute_r_squared(adc_22_07_df, 'yield', 'yield_pred_2022')         # R2 = 0.5 -- 22CA
# compute_r_squared(adc_22_07_df, 'yield_irrig', 'yield_pred_2022')   # R2 = 0.4 -- 22CA
# compute_r_squared(adc_22_07_df, 'yield_rf', 'yield_pred_2022')      # R2 = 0.28 -- 22CA
# compute_r_squared(adc_22_07_df, 'yield_oi', 'yield_pred_2022')      # R2 = 0.74 -- 22CA
# compute_r_squared(adc_22_07_df, 'yield_pv', 'yield_pred_2022')      # R2 = 0.5 -- 22CA
# compute_r_squared(adc_22_07_df, 'yield_gp', 'yield_pred_2022')      # R2 = 0.52 -- 22CA
# compute_r_squared(adc_22_07_df, 'yield_no_gp', 'yield_pred_2022') # R2 = 0.5 -- 22CA
# compute_r_squared(adc_22_07_df, 'yield', 'yield_siap')            # R2 = 0.58 -- 22CA
# compute_r_squared(adc_22_07_df, 'yield_rf', 'yield_siap')         # R2 = 0.32 -- 22CA
# compute_r_squared(adc_22_07_df, 'yield_oi', 'yield_siap')         # R2 = 0.88 -- 22CA


In [ ]:
# ### Compute R2 for CA2022 data at municipality level
# ca22_df = pd.read_stata(mun_22_data)
# ca22_df = ca22_df[ca22_df['name'] == 'Maize']
# ca22_df = ca22_df[['name', 'muncode','share_irrig']+[c for c in ca22_df.columns if 'yield' in c]]
# ### Read seasonal data
# ca22_df_szn = pd.read_stata(mun_22_szn)
# ca22_df_szn = ca22_df_szn[ca22_df_szn['name'] == 'Maize']
# ca22_oi_df  = ca22_df_szn[ca22_df_szn['type'] == "o-i"]
# ca22_oi_df  = ca22_oi_df[['name', 'muncode']+[c for c in ca22_oi_df.columns if 'yield' in c]]
# ca22_oi_df.columns = ['name', 'muncode']+[c.replace('yield', 'yield_oi') for c in ca22_oi_df.columns if 'yield' in c]
# ca22_pv_df  = ca22_df_szn[ca22_df_szn['type'] == "p-v"]
# ca22_pv_df  = ca22_pv_df[['name', 'muncode']+[c for c in ca22_pv_df.columns if 'yield' in c]]
# ca22_pv_df.columns = ['name', 'muncode']+[c.replace('yield', 'yield_pv') for c in ca22_pv_df.columns if 'yield' in c]
# ca22_df = ca22_df.merge(ca22_oi_df, on=['name', 'muncode'], how='outer').merge(ca22_pv_df, on=['name', 'muncode'], how='outer')
# ca22_df['muncode'] = ca22_df['muncode'].astype(int)
# ### Merge with SIAP
# siap_df          =  pd.read_stata(siap_data)
# siap_df          =  siap_df[siap_df['name'] == 'Maize']
# siap_df['yield'] =  siap_df['q']/siap_df['ha_planted']
# siap_df['muncode'] = siap_df['muncode'].astype(int)
# siap_df          =  siap_df[siap_df['year'] == 2022]
# siap_df = siap_df[['muncode','lshare','yield']]
# siap_df.columns = ['muncode', 'share_crop', 'yield_siap']
# ca22_df = ca22_df.merge(siap_df, on='muncode', how='outer')
# ### Read in Joel predictions
# mun22_pred_df = pd.read_csv(mun_2022_pred)
# mun22_pred_df = mun22_pred_df[['muncode', 'yield_pred_ls']]
# ### Merge all together
# ca22_df = ca22_df.merge(mun22_pred_df, on='muncode', how='inner')

In [ ]:
# compute_r_squared(ca22_df, 'yield_siap', 'yield') # R2 = 0.704 -- Correlation between mun level CA22 and SIAP22
# compute_r_squared(ca22_df, 'yield_siap', 'yield_oi') # R2 = 0.56 -- Correlation between mun level CA22 (fall-winter) and SIAP22
# compute_r_squared(ca22_df, 'yield_siap', 'yield_pv') # R2 = 0.692 -- Correlation between mun level CA22 (spring-summer) and SIAP22

### Correlations between predictions and yields in SIAP22 and CA22
# compute_r_squared(ca22_df, 'yield_siap', 'yield_pred_ls') # R2 = 0.149 -- 22 SIAP
# compute_r_squared(ca22_df, 'yield', 'yield_pred_ls') # R2 = 0.131 -- 22CA
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.2], 'yield_siap', 'yield_pred_ls')  # R2 = 0.219 -- 22 SIAP
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.2], 'yield', 'yield_pred_ls')       # R2 = 0.225 -- 22CA
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_siap', 'yield_pred_ls')  # R2 = 0.276 -- 22 SIAP
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield', 'yield_pred_ls')       # R2 = 0.324 -- 22CA
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.75], 'yield_siap', 'yield_pred_ls') # R2 = 0.208 -- 22 SIAP
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.75], 'yield', 'yield_pred_ls')      # R2 = 0.307 -- 22CA
# compute_r_squared(ca22_df[ca22_df['share_crop'] == 1], 'yield_siap', 'yield_pred_ls')   # R2 = 0.139 -- 22 SIAP
# compute_r_squared(ca22_df[ca22_df['share_crop'] == 1], 'yield', 'yield_pred_ls')        # R2 = 0.096 -- 22CA

# compute_r_squared(ca22_df, 'yield_pred_ls', 'yield_irrig')                              # R2 = 0.057 -- 22CA, only irrig plot
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_irrig') # R2 = 0.156 -- 22CA, only irrig plot
# compute_r_squared(ca22_df, 'yield_pred_ls', 'yield_rf')                                 # R2 = 0.378 -- 22CA, only rf plot
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_rf')    # R2 = 0.485 -- 22CA, only rf plot
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_gp')    # R2 = 0.255 -- 22CA, only large producers
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_no_gp') # R2 = 0.318 -- 22CA, only small producers

# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_oi')       # R2 = 0.097 -- 22CA, Fall-Winter
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_pv')       # R2 = 0.331 -- 22CA, Spring-Summer
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_oi_irrig') # R2 = 0.064 -- 22CA, Fall-Winter, only irrig plot
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_oi_rf')    # R2 = 0.115 -- 22CA, Fall-Winter, only rf plot
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_pv_irrig') # R2 = 0.147 -- 22CA, Spring-Summer, only irrig plot
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_pv_rf')    # R2 = 0.532 -- 22CA, Spring-Summer, only rf plot

# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_oi_gp')    # R2 = 0.047 -- 22CA, Fall-Winter, only large producers
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_oi_no_gp') # R2 = 0.099 -- 22CA, Fall-Winter, only small producers

# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_pv_gp')    # R2 = 0.295 -- 22CA, Spring-Summer, only large producers
# compute_r_squared(ca22_df[ca22_df['share_crop'] > 0.5], 'yield_pred_ls', 'yield_pv_no_gp') # R2 = 0.326 -- 22CA, Spring-Summer, only small producers

In [ ]:
### Parameters
year_analyze      = 2007
adc_ref_year      = 2007
use_amca_sup_sum  = False
season_comp       = "combined" ##"o-i" ###"p-v"
adc_preds_version = "old" ### "new" ### "old"

if season_comp   != "combined":
    siap_data     =  os.path.join(siap_dir,   "siap_ag_prod_estimation_by_season.dta")


In [ ]:
### Read in comparable yields 
siap_df          =  pd.read_stata(siap_data)
siap_df          =  siap_df[siap_df['name'] == 'Maize']
if season_comp == "o-i":
    siap_df          =  siap_df[siap_df['growing_season'] == 'Fall-Winter']
elif season_comp == "p-v":
    siap_df          =  siap_df[siap_df['growing_season'] == 'Spring-Summer']
siap_df['yield'] =  siap_df['q']/siap_df['ha_planted']
siap_df['muncode'] = siap_df['muncode'].astype(int)
siap_df          =  siap_df[siap_df['year'] == year_analyze]
mun_pred_df = pd.read_csv(mun_preds)
mun_pred_df = mun_pred_df[['muncode','year','pred']]
mun_pred_df = mun_pred_df.merge(siap_df[['muncode','year','yield']], on=['muncode','year'], how='left')
# for yr in range(2003,2021):
    # print(yr, compute_r_squared(mun_pred_df[mun_pred_df['year'] == yr], 'pred', 'yield'))
if year_analyze < 2021: mun_pred_df = mun_pred_df[mun_pred_df['year'] == year_analyze]
else: mun_pred_df = mun_pred_df[mun_pred_df['year'] == 2020]
siap_df          =  siap_df[['muncode','yield','ha_planted']].reset_index(drop=True)
siap_df.columns = ['muncode','munyield','ha_planted']


In [ ]:
if adc_ref_year == 2016:
    ### Read in 2007 ADC yield data from 2016 ADCs
    adc_df = pd.read_stata(adc_07_2016).drop('cycle',axis=1)
    # ### To sum across seasons -- remove to separate seasons
    adc_df = adc_df[adc_df['type'] == 'total']
    # ### To sum across seasons -- remove to separate seasons
    # adc_df = adc_df[adc_df['type'] != 'total'] ### separate seasons
    adc_df = adc_df.rename(columns={'adc':'adcid'})
    adc_df['muncode'] = adc_df['muncode'].astype(int)
    if use_amca_sup_sum:
        amca_df = pd.read_csv(adc_16_data)
        amca_df = amca_df[amca_df['name'] == 'Maize']
        amca_df = amca_df[['adc','sup_sem','num_terrenos']]
        amca_df.columns = ['adcid','sup_sem','num_terrenos']
        adc_df = adc_df.rename(columns={'sup_sem':'sup_sem_07','num_terrenos':'num_terrenos_07'})
        adc_df = adc_df.merge(amca_df, on = 'adcid', how = 'outer')
        adc_df['type'] = 'total'
        adc_df['name'] = 'Maize'
        adc_df['year'] = 2007
        adc_df['muncode'] = adc_df['adcid'].apply(lambda x: x[:5]).astype(int)
        adc_df['ageb'] = adc_df['adcid'].apply(lambda x: x[:10])
        adc_df['cve_ent'] = adc_df['adcid'].apply(lambda x: x[:2])
        adc_df['sup_sem'] = adc_df['sup_sem'].fillna(adc_df['sup_sem_07'])
        adc_df['num_terrenos'] = adc_df['num_terrenos'].fillna(adc_df['num_terrenos_07'])
        adc_df = adc_df.drop(['sup_sem_07','num_terrenos_07'],axis=1)

else:
    ## Read in ADC yield data from 2007 ADCs
    adc_df = pd.read_stata(adc_07_data)[['adc','name','type','muncode','CVE_ENT','num_terrenos','sup_sem','Q','yield']]
    if season_comp == "combined":
        ### To sum across seasons -- remove to separate seasons
        adc_df['type'] = 'combined'
        adc_df = adc_df.groupby(['adc','name','type','muncode','CVE_ENT'])[['num_terrenos','sup_sem','Q']].sum().reset_index()
        adc_df['yield'] = adc_df['Q']/adc_df['sup_sem']
        ### To sum across seasons -- remove to separate seasons
    adc_df.columns = ['adcid','name','type','muncode','CVE_ENT','num_terrenos','sup_sem','Q','yield']
    ### Compute total plantings in ADCs
    total_adc_plantings = adc_df.groupby(['adcid'])['sup_sem'].sum().reset_index()
    total_adc_plantings.columns = ['adcid','total_plantings']
    adc_df = adc_df.merge(total_adc_plantings, on = 'adcid', how = 'left')
    adc_df = adc_df[adc_df['name'] == 'Maize'] ### 'Avocados'
    adc_df['share_maize'] = adc_df['sup_sem']/adc_df['total_plantings']
    ### Subset down to season of interest
    adc_df = adc_df[adc_df['type'] == season_comp]
    


In [ ]:
### Compute R2 between yields from ADC data -- (adc level, ageb. avg. lvl, and mun. avg. lvl) with SIAP data
# adc_df['ageb'] = adc_df['adcid'].apply(lambda x: x[:10])
# ageb_avg_adc_yield_df = adc_df.groupby(['muncode','ageb'])[['sup_sem','Q']].sum().reset_index()
# ageb_avg_adc_yield_df['ageb_adc_avg_yield'] = ageb_avg_adc_yield_df['Q']/ageb_avg_adc_yield_df['sup_sem']
# ageb_avg_adc_yield_df = ageb_avg_adc_yield_df[['muncode','ageb','ageb_adc_avg_yield','sup_sem']]
# ageb_avg_adc_yield_df = ageb_avg_adc_yield_df.merge(siap_df.drop('ha_planted',axis=1), on = 'muncode', how = 'left') ### Merge with SIAP data


# mun_avg_adc_yield_df = adc_df.groupby(['muncode'])[['sup_sem','Q']].sum().reset_index()
# mun_avg_adc_yield_df['mun_adc_avg_yield'] = mun_avg_adc_yield_df['Q']/mun_avg_adc_yield_df['sup_sem']
# mun_avg_adc_yield_df = mun_avg_adc_yield_df[['muncode','mun_adc_avg_yield','sup_sem']]
# mun_avg_adc_yield_df = mun_avg_adc_yield_df.merge(siap_df.drop('ha_planted',axis=1), on = 'muncode', how = 'left') ### Merge with SIAP data

# adc_df = adc_df.merge(siap_df.drop('ha_planted',axis=1), on = 'muncode', how = 'left') ### Merge with SIAP data


# compute_r_squared(adc_df, 'yield', 'munyield'), compute_r_squared(ageb_avg_adc_yield_df, 'ageb_adc_avg_yield', 'munyield'), compute_r_squared(mun_avg_adc_yield_df, 'mun_adc_avg_yield', 'munyield') # R2 = 0.46



In [ ]:
### Compute municipality average yield from ADC data for avocados
mun_avg_adc_yield_df = adc_df.groupby(['muncode','name','CVE_ENT'])[['sup_sem','Q',]].sum().reset_index()
mun_avg_adc_yield_df['yield'] = mun_avg_adc_yield_df['Q']/mun_avg_adc_yield_df['sup_sem']
mun_avg_adc_yield_df = mun_avg_adc_yield_df[['muncode','name','CVE_ENT','yield','Q','sup_sem']]
mun_avg_adc_yield_df = mun_avg_adc_yield_df.to_csv(mun_yield_07, index=False)


In [ ]:
### Compute municipality average yield from ADC data
mun_avg_adc_yield_df = adc_df.groupby(['muncode'])[['sup_sem','Q']].sum().reset_index()
mun_avg_adc_yield_df['mun_adc_avg_yield'] = mun_avg_adc_yield_df['Q']/mun_avg_adc_yield_df['sup_sem']
mun_avg_adc_yield_df = mun_avg_adc_yield_df[['muncode','mun_adc_avg_yield','sup_sem']]

# ### Read in predicted yields
if adc_preds_version == "old":
    adc_pred_df = pd.read_csv(adc_preds)[['adcid','yield_pred_ls']]
elif adc_preds_version == "new":
    ### Read in Joel's predictions
    adc_pred_df = pd.read_csv(adc_2024)
    adc_pred_df = adc_pred_df[adc_pred_df['year'] == year_analyze].drop('year',axis=1)
    adc_pred_df.columns = ['adcid','yield_pred_ls']
    # ### Write out the 2022 predictions
    if year_analyze == 2022:
        adc_pred_df['year'] = 2022
        adc_pred_df.sort_values('adcid').to_csv(adc_2022_pred, index=False)
adc_df = adc_df.merge(adc_pred_df, on = 'adcid', how='outer') ### Merge with predicted yields
### Read in predicted hectares
### Read in ADC ha planted predictions
ha_preds_df = pd.read_csv(ha_preds)
ha_preds_df.columns = ['adcid','ha_pred']
adc_df = adc_df.merge(ha_preds_df, on = 'adcid', how='left') ### Merge with predicted hectares
adc_df = adc_df.merge(siap_df.drop('ha_planted',axis=1), on = 'muncode', how = 'left') ### Merge with SIAP data
if adc_ref_year == 2016:
    land_area_df = pd.read_csv(adc_land_area_16)
    land_area_df = land_area_df.rename(columns={'adc16':'adcid'})
elif adc_ref_year == 2007: 
    land_area_df = pd.read_csv(adc_land_area_07)
    pass  # agland CSV already ships an 'adcid' column
land_area_df = land_area_df[['adcid','adc_area','siap_agland_area','inegi_agland_area','maize_land']]
land_area_df.columns = ['adcid','adc_area','siap_ag_area','inegi_ag_area','siap_mz_area']
adc_df = adc_df.merge(land_area_df, on = 'adcid', how='left') ### Merge with ADC ag land estimates
adc_df = adc_df.merge(mun_avg_adc_yield_df.drop('sup_sem',axis=1), on = 'muncode', how='left') ### Merge with CA2007 mun yield estimates

### Compute predicted hectares at mun. level
ha_preds_df['muncode'] = ha_preds_df['adcid'].str[:5].astype(int)
ha_pred_mun_df = ha_preds_df.groupby('muncode')['ha_pred'].sum().reset_index()

# mun_yield_df = adc_df.groupby('muncode')[['sup_sem','Q']].sum().reset_index()
# mun_yield_df['mun_yield'] = mun_yield_df['Q']/mun_yield_df['sup_sem']
# mun_yield_df['lmun_yield'] = mun_yield_df['mun_yield'].apply(np.log)
# mun_yield_df = mun_yield_df.drop(['sup_sem','Q'], axis=1)
# mun_yield_df['muncode'] = mun_yield_df['muncode'].astype(str)
# adc_df = adc_df.drop(['name','muncode','CVE_ENT'], axis=1)

In [ ]:
### Compute avg. yield based on ADC predictions and different weighting schemes
### And use this to compute ex-post correction
weight_schemes = ['sup_sem','adc_area','siap_ag_area','inegi_ag_area','siap_mz_area','ha_pred']
for sum_var in weight_schemes:
   adc_df['pred_Q_'+sum_var]    =  adc_df['yield_pred_ls'] * adc_df[sum_var] ### Predicted yield * area
   adc_df['to_sum_'+sum_var]       =  adc_df.apply(lambda x: x[sum_var]     if not np.isnan(x['yield_pred_ls']) else 0, axis = 1)


adc_df['to_sum_Q']              =  adc_df.apply(lambda x: x['Q']           if not np.isnan(x['yield_pred_ls']) else 0, axis = 1)
adc_df['num_adcs']              =  adc_df.apply(lambda x: 1 if not np.isnan(x['yield_pred_ls']) else 0, axis = 1)

mun_sum_cols = ['pred_Q_'+wght for wght in weight_schemes]
mun_sum_cols = mun_sum_cols+['Q','sup_sem','ha_pred','num_adcs','to_sum_Q']
mun_sum_cols = mun_sum_cols+['to_sum_'+wght for wght in weight_schemes]

avg_yield_pred_df = adc_df.groupby('muncode')[mun_sum_cols].sum().reset_index()

for sum_var in weight_schemes:
   avg_yield_pred_df['yield_pred_'+sum_var] = avg_yield_pred_df['pred_Q_'+sum_var]/avg_yield_pred_df['to_sum_'+sum_var]

avg_yield_pred_df['yield_agg_exceptions']   = avg_yield_pred_df['to_sum_Q']/avg_yield_pred_df['to_sum_sup_sem']
avg_yield_pred_df['yield_real']             = avg_yield_pred_df['Q']/avg_yield_pred_df['sup_sem']

avg_yield_pred_cols = ['muncode','yield_agg_exceptions','yield_real','ha_pred','num_adcs']+['yield_pred_'+wght for wght in weight_schemes]
avg_yield_pred_df = avg_yield_pred_df[avg_yield_pred_cols]
avg_yield_pred_df = avg_yield_pred_df.merge(siap_df, on = 'muncode', how = 'left') ### SIAP mun. avg. yield 
avg_yield_pred_df = avg_yield_pred_df.merge(mun_avg_adc_yield_df, on = 'muncode', how = 'left') ### Treat agg. CA data as SIAP mun. avg. yield.


for sum_var in weight_schemes:
   avg_yield_pred_df['diff_'+sum_var]     = avg_yield_pred_df['yield_pred_'+sum_var] - avg_yield_pred_df['munyield']
   avg_yield_pred_df['truediff_'+sum_var] = avg_yield_pred_df['yield_pred_'+sum_var] - avg_yield_pred_df['mun_adc_avg_yield']
   avg_yield_pred_df['diff_ratio_'+sum_var]    = avg_yield_pred_df['yield_pred_'+sum_var]/avg_yield_pred_df['munyield']
   avg_yield_pred_df['truediff_ratio_'+sum_var]= avg_yield_pred_df['yield_pred_'+sum_var]/avg_yield_pred_df['mun_adc_avg_yield']

# avg_yield_pred_df['diff_hapred']     = (avg_yield_pred_df['ha_pred'] - avg_yield_pred_df['ha_planted'])/avg_yield_pred_df['num_adcs']
# avg_yield_pred_df['truediff_hapred'] = (avg_yield_pred_df['ha_pred'] - avg_yield_pred_df['sup_sem'])/avg_yield_pred_df['num_adcs']

tosub_df = avg_yield_pred_df[['muncode']+[col for col in avg_yield_pred_df.columns if 'diff_' in col]]
adc_rc_df = adc_df.merge(tosub_df, on = 'muncode', how = 'left')

In [ ]:
### Compute correlations between uncorrected yield predictions with different weightings and SIAP mun yield
# compute_r_squared(avg_yield_pred_df, 'yield_pred_sup_sem', 'munyield')       # R2 = 0.336 -- 07CA, 0.54  -- 16AMCA
# compute_r_squared(avg_yield_pred_df, 'yield_pred_adc_area', 'munyield')      # R2 = 0.293 -- 07CA, 0.535 -- 16AMCA
# compute_r_squared(avg_yield_pred_df, 'yield_pred_siap_ag_area', 'munyield')  # R2 = 0.342 -- 07CA, 0.536 -- 16AMCA
# compute_r_squared(avg_yield_pred_df, 'yield_pred_inegi_ag_area', 'munyield') # R2 = 0.344 -- 07CA, 0.543 -- 16AMCA
# compute_r_squared(avg_yield_pred_df, 'yield_pred_siap_mz_area', 'munyield')  # R2 = 0.322 -- 07CA, 0.593 -- 16AMCA
# compute_r_squared(avg_yield_pred_df, 'yield_pred_ha_pred', 'munyield')       # R2 = 0.28  -- 07CA, 0.512 -- 16AMCA

### Compute R2 between yield predictions with weighting schemes and "true" weights -- i.e. sup_sum from census
# compute_r_squared(avg_yield_pred_df, 'yield_pred_sup_sem', 'yield_pred_adc_area')      # R2 = 0.82  -- 07CA, 0.932 -- 16AMCA
# compute_r_squared(avg_yield_pred_df, 'yield_pred_sup_sem', 'yield_pred_siap_ag_area')  # R2 = 0.862 -- 07CA, 0.906 -- 16AMCA
# compute_r_squared(avg_yield_pred_df, 'yield_pred_sup_sem', 'yield_pred_inegi_ag_area') # R2 = 0.868 -- 07CA, 0.901 -- 16AMCA
# compute_r_squared(avg_yield_pred_df, 'yield_pred_sup_sem', 'yield_pred_siap_mz_area')  # R2 = 0.72  -- 07CA, 0.862 -- 16AMCA
# compute_r_squared(avg_yield_pred_df, 'yield_pred_sup_sem', 'yield_pred_ha_pred')       # R2 = 0.8   -- 07CA, 0.88  -- 16AMCA

### Compute actual correlation between real yield and municipality level yield from SIAP
# compute_r_squared(avg_yield_pred_df, 'yield_real', 'munyield') # R2 = 0.49 -- 2007 CA
# avg_yield_pred_df['CVE_ENT'] = avg_yield_pred_df['muncode'].apply(lambda x: int(x/1000))
# compute_r_squared(avg_yield_pred_df[avg_yield_pred_df['CVE_ENT'].isin([2,3,25,26,28])], 'yield_real', 'munyield')  # R2 = 0.56 -- 07CA
# compute_r_squared(avg_yield_pred_df[~avg_yield_pred_df['CVE_ENT'].isin([2,3,25,26,28])], 'yield_real', 'munyield') # R2 = 0.46 -- 07CA

### Compute R2 between real yield (where predictions exist) and predicted yield at mun level
# compute_r_squared(avg_yield_pred_df, 'yield_agg_exceptions', 'yield_pred_sup_sem')       # R2 = 0.285 -- 07CA
# compute_r_squared(avg_yield_pred_df, 'yield_agg_exceptions', 'yield_pred_adc_area')      # R2 = 0.23  -- 07CA
# compute_r_squared(avg_yield_pred_df, 'yield_agg_exceptions', 'yield_pred_siap_ag_area')  # R2 = 0.288 -- 07CA
# compute_r_squared(avg_yield_pred_df, 'yield_agg_exceptions', 'yield_pred_inegi_ag_area') # R2 = 0.287 -- 07CA
# compute_r_squared(avg_yield_pred_df, 'yield_agg_exceptions', 'yield_pred_siap_mz_area')  # R2 = 0.274 -- 07CA
# compute_r_squared(avg_yield_pred_df, 'yield_agg_exceptions', 'yield_pred_ha_pred')       # R2 = 0.26  -- 07CA



In [ ]:
for sum_var in weight_schemes:
    ### Correct by ADC avg. SIAP yield
    adc_rc_df['corr_yield_pred_'+sum_var] = adc_rc_df['yield_pred_ls']-adc_rc_df['diff_'+sum_var]
    adc_rc_df['corr_yield_pred_'+sum_var] = adc_rc_df['corr_yield_pred_'+sum_var].apply(lambda x: 0 if x < 0 else x)
    adc_rc_df['corr_yield_pred_'+sum_var] = adc_rc_df.apply(lambda x: x['corr_yield_pred_'+sum_var] if not np.isnan(x['yield_pred_ls']) else np.nan, axis = 1)
   
    ### Correct by true ADC avg. yield
    adc_rc_df['ca_avg_corr_yield_'+sum_var] = adc_rc_df['yield_pred_ls']-adc_rc_df['truediff_'+sum_var]
    adc_rc_df['ca_avg_corr_yield_'+sum_var] = adc_rc_df['ca_avg_corr_yield_'+sum_var].apply(lambda x: 0 if x < 0 else x)
    adc_rc_df['ca_avg_corr_yield_'+sum_var] = adc_rc_df.apply(lambda x: x['ca_avg_corr_yield_'+sum_var] if not np.isnan(x['yield_pred_ls']) else np.nan, axis = 1)

    ### Correct by ADC avg. SIAP yield (mult)
    adc_rc_df['corr_yield_scl_pred_'+sum_var] = adc_rc_df['yield_pred_ls']/adc_rc_df['diff_ratio_'+sum_var]
    adc_rc_df['ca_avg_corr_yield_scl_pred_'+sum_var] = adc_rc_df['yield_pred_ls']/adc_rc_df['truediff_ratio_'+sum_var]

### Correct HA preds by mun SIAP hectares planted on average
# adc_rc_df['ha_rc_pred'] = adc_rc_df['ha_pred']-adc_rc_df['diff_hapred']
# adc_rc_df['ha_rc_pred'] = adc_rc_df['ha_rc_pred'].apply(lambda x: 0 if x < 0 else x)
# adc_rc_df['ha_rc_pred'] = adc_rc_df.apply(lambda x: x['ha_rc_pred'] if not np.isnan(x['yield_pred_ls']) else np.nan, axis = 1)

# ### Correct HA preds by true ADC avg. hectares planted
# adc_rc_df['ha_adcc_pred'] = adc_rc_df['ha_pred']-adc_rc_df['truediff_hapred']
# adc_rc_df['ha_adcc_pred'] = adc_rc_df['ha_adcc_pred'].apply(lambda x: 0 if x < 0 else x)
# adc_rc_df['ha_adcc_pred'] = adc_rc_df.apply(lambda x: x['ha_adcc_pred'] if not np.isnan(x['yield_pred_ls']) else np.nan, axis = 1)

In [ ]:
### Write ex-post corrections to file
if year_analyze == 2022:
    ex_post_corr_df = adc_rc_df[['adcid','yield_pred_ls']+[col for col in adc_rc_df.columns if 'corr_yield' in col and (('ca_avg' not in col) and ('sup_sem' not in col))]]
    ex_post_corr_df['year'] = 2022
    ex_post_corr_df.sort_values('adcid').to_csv(adc_expost_corr_pred_22, index=False)
elif year_analyze == 2007:
    ex_post_corr_df = adc_rc_df[['adcid','yield_pred_ls']+[col for col in adc_rc_df.columns if 'corr_yield' in col and (('ca_avg' not in col) and ('sup_sem' not in col))]]
    ex_post_corr_df['year'] = 2007
    ex_post_corr_df.sort_values('adcid').to_csv(adc_expost_corr_pred_07, index=False)


In [ ]:
fav_wght = 'siap_ag_area'

In [ ]:
adc_rc_df['yield_pred_ls'].std(), adc_rc_df['corr_yield_scl_pred_'+fav_wght].std(), adc_rc_df['corr_yield_pred_'+fav_wght].std(),  adc_rc_df['yield'].std()


In [ ]:
plt.scatter(adc_rc_df['corr_yield_scl_pred_siap_ag_area'], adc_rc_df['yield'])
plt.xlabel('yield_rc_pred_mult')
plt.ylabel('yield')
plt.title('Scatter Plot')
# plt.xlim(0, 15)
# plt.ylim(0, 15)
plt.show()


In [ ]:
compute_r_squared(adc_df[adc_df['adc_area'] < 0.1], 'yield', 'yield_pred_ls')                      ### 0.088 -- 07CA
compute_r_squared(adc_df[adc_df['CVE_ENT'].isin(['02','03','25','26','28'])], 'yield', 'munyield') ### 0.517 -- 07CA


In [ ]:
# compute_r_squared(adc_rc_df[adc_rc_df['share_maize'] >  0.1], 'yield', 'corr_yield_pred_'+fav_wght)  ### 0.304
# compute_r_squared(adc_rc_df[adc_rc_df['share_maize'] <= 0.1], 'yield', 'corr_yield_pred_'+fav_wght)  ### 0.191
# compute_r_squared(adc_rc_df[adc_rc_df['share_maize'] >  0.5], 'yield', 'corr_yield_pred_'+fav_wght)  ### 0.31
# compute_r_squared(adc_rc_df[adc_rc_df['share_maize'] <= 0.5], 'yield', 'corr_yield_pred_'+fav_wght)  ### 0.268

adc_median = adc_rc_df['adc_area'].median()
compute_r_squared(adc_rc_df[adc_rc_df['adc_area'] <= adc_median], 'yield', 'corr_yield_pred_'+fav_wght)     ### 0.274 -- 07CA
compute_r_squared(adc_rc_df[adc_rc_df['adc_area'] > adc_median], 'yield', 'corr_yield_pred_'+fav_wght)      ### 0.323 -- 07CA
compute_r_squared(adc_rc_df[adc_rc_df['adc_area'] <= adc_median], 'yield', 'corr_yield_scl_pred_'+fav_wght) ### 0.204 -- 07CA
compute_r_squared(adc_rc_df[adc_rc_df['adc_area'] > adc_median], 'yield', 'corr_yield_scl_pred_'+fav_wght)  ### 0.276 -- 07CA


In [ ]:
# (leftover interactive inspection of `adcid`; neutralized 2026-08-28)


In [ ]:
### Compare yield predictions to real yields in 2007
# compute_r_squared(adc_rc_df, 'yield', 'yield_pred_ls')                                 ### 0.15  - 07CA/o-i, 0.13 - 07CA/combined
# compute_r_squared(adc_rc_df, 'yield', 'corr_yield_pred_'+fav_wght)                     ### 0.347 - 07CA/o-i, 0.293 - 07CA/combined
# compute_r_squared(adc_rc_df[adc_rc_df['share_maize'] > 0.2], 'yield', 'yield_rc_pred') ### 0.59 for o-i (ADC) SIAP full, 0.305 for combined
# compute_r_squared(adc_rc_df, 'yield', 'munyield')                                      ### 0.364 for o-i, 0.245 for combined
# compute_r_squared(adc_rc_df, 'yield', 'mun_adc_avg_yield')                             ### 0.357 for combined

# compute_r_squared(adc_rc_df[adc_rc_df['share_maize'] > 0.1], 'yield', 'munyield')      ### 0.594 for o-i, 0.263 for combined

# compute_r_squared(adc_rc_df[adc_rc_df['CVE_ENT'].isin(['02','03','25','26','28'])], 'yield', 'yield_rc_pred') ### 0.4647 for combined
# compute_r_squared(adc_rc_df[~adc_rc_df['CVE_ENT'].isin(['02','03','25','26','28'])], 'yield', 'yield_rc_pred') ### 0.216 for combined
# compute_r_squared(adc_rc_df, 'yield', 'yield_adcc_pred') ### 0.432 for combined
# compute_r_squared(adc_rc_df, 'yield', 'yield_adcc_pred_mult') ### 0.432 for combined
# compute_r_squared(adc_rc_df[adc_rc_df['CVE_ENT'].isin(['25','28'])], 'yield', 'yield_rc_pred') ### 0.4906 for combined

# Sinaloa and Tam

In [ ]:
compute_r_squared(avg_yield_pred_df, 'munyield', 'yield_real') # R2 = 0.4893
compute_r_squared(avg_yield_pred_df, 'yield_agg_exceptions', 'yield_pred_agland_area') # R2 = 0.288

compute_r_squared(avg_yield_pred_df, 'yield_real', 'yield_pred_agland_area') # R2 = 0.289



In [ ]:
for sum_var in weight_schemes:
    adc_rc_df['pred_Q_'+sum_var]     =  adc_rc_df['corr_yield_pred_'+sum_var]     * adc_rc_df[sum_var]
    adc_rc_df['pred_Q_scl_'+sum_var] =  adc_rc_df['corr_yield_scl_pred_'+sum_var] * adc_rc_df[sum_var]
    adc_rc_df['to_sum_'+sum_var]     =  adc_rc_df.apply(lambda x: x[sum_var] if not np.isnan(x['corr_yield_pred_'+sum_var]) else 0, axis = 1)
    adc_rc_df['to_sum_Q_'+sum_var]   =  adc_rc_df.apply(lambda x: x['Q']     if not np.isnan(x['corr_yield_pred_'+sum_var]) else 0, axis = 1)

avg_yield_rc_pred_gcols = ['Q','sup_sem']+['pred_Q_'+w for w in weight_schemes]+['pred_Q_scl_'+w for w in weight_schemes]+['to_sum_'+w for w in weight_schemes]+['to_sum_Q_'+w for w in weight_schemes]
avg_yield_rc_pred_df = adc_rc_df.groupby('muncode')[avg_yield_rc_pred_gcols].sum().reset_index()

for sum_var in weight_schemes:
    for scl_term in ['','scl_']:
        avg_yield_rc_pred_df['yield_pred_'+scl_term+sum_var] = avg_yield_rc_pred_df['pred_Q_'+scl_term+sum_var]/adc_rc_df['to_sum_'+sum_var]
    avg_yield_rc_pred_df['yield_agg_excep_'+sum_var]   = avg_yield_rc_pred_df['to_sum_Q_'+sum_var]/avg_yield_rc_pred_df['to_sum_sup_sem']

avg_yield_rc_pred_df['yield_real']             = avg_yield_rc_pred_df['Q']/avg_yield_rc_pred_df['sup_sem']

avg_yield_rc_pred_df = avg_yield_rc_pred_df[['muncode']+[c for c in avg_yield_rc_pred_df.columns if 'yield' in c]]
avg_yield_rc_pred_df = avg_yield_rc_pred_df.merge(siap_df, on = 'muncode', how = 'left')


In [ ]:
compute_r_squared(avg_yield_rc_pred_df, 'yield_real', 'yield_pred_siap_ag_area') # R2 = 0.346


In [ ]:
# compute_r_squared(adc_df, 'yield', 'munyield') ## 0.245

# compute_r_squared(adc_df, 'yield', 'yield_pred_ls') ### 0.13
compute_r_squared(adc_df[adc_df['share_maize'] > 0.20], 'yield', 'munyield') ### 0.267

compute_r_squared(adc_df[adc_df['share_maize'] > 0.20], 'yield', 'yield_pred_ls') ### 0.14
# compute_r_squared(adc_df[adc_df['share_maize']== 1], 'yield', 'munyield')

# compute_r_squared(adc_df[adc_df['share_maize'] == 1], 'yield', 'yield_pred_ls')

compute_r_squared(adc_df[adc_df['CVE_ENT'].isin([2,3,25,26,28])], 'yield', 'munyield') ### 0.516
# compute_r_squared(adc_df[~adc_df['CVE_ENT'].isin([2,3,25,26,28])], 'yield', 'munyield') ### 0.172

compute_r_squared(adc_df[adc_df['CVE_ENT'].isin([2,3,25,26,28])], 'yield', 'yield_pred_ls') ### 0.09
# compute_r_squared(adc_df[~adc_df['CVE_ENT'].isin([2,3,25,26,28])], 'yield', 'yield_pred_ls') ### 0.107


In [ ]:
### Read in ADC shapefile for plotting
adc07_df = gpd.read_file(map_adc_path) ### ADC shapefile
adc07_df.set_geometry('geometry', inplace=True)

### Create municipality shapefile from ADC07
# mundf_from_adc = adc07_df[~adc07_df['adcid'].isna()]
# mundf_from_adc.loc[:,'muncode'] = mundf_from_adc['est']+mundf_from_adc['mun']
# mundf_from_adc = mundf_from_adc.dissolve(by='muncode')
# mundf_from_adc = mundf_from_adc.reset_index()[['geometry','muncode']]
# mundf_from_adc.to_file(map_mun_path_ca2007, index=False)

### Read in ADC16 shapefile
adc_shapefile = gpd.read_file(adc_2016_shp) ### 2016 ADC shapefile
adc_shapefile = adc_shapefile[['CONTROL','geometry']]
adc_shapefile.columns = ['adcid','geometry']
adc_shapefile.set_geometry('geometry', inplace=True)
adc_shapefile = adc_shapefile.set_crs(adc07_df.crs)

# len(adc_shapefile[adc_shapefile['ha_area'] > 0.0003])/len(adc_shapefile)  # ha_area was interactive-only

### Produce ADC/mun shapefile size stats
# adc_shapefile[adc_shapefile['ha_area']==adc_shapefile['ha_area'].max()]
# len(adc_shapefile[adc_shapefile['ha_area'] < municipio_shapefile['ha_area'].min()])/len(adc_shapefile)

adc_shapefile = adc_shapefile.to_crs(str(st_line_shp.crs))

### For subsetting down to a region of interest
# st_line_shp = st_line_shp[st_line_shp['CVE_ENT'] == '20']
### ytop, ybottom = 18.8, 18.1
### xleft, xright = -96.9, -96.1

### adc_shapefile = adc_shapefile[adc_shapefile['est'] == '20']

# ### Subset down to region around 20002
# adc_shapefile = adc_shapefile[adc_shapefile['repy'] < ytop]
# adc_shapefile = adc_shapefile[adc_shapefile['repy'] > ybottom]
# adc_shapefile = adc_shapefile[adc_shapefile['repx'] > xleft]
# adc_shapefile = adc_shapefile[adc_shapefile['repx'] < xright]

In [ ]:
adc_shapefile = adc_shapefile.merge(adc_rc_df, on = 'adcid', how = 'left')
adc_shapefile['lyield'] = adc_shapefile['yield'].apply(np.log)
adc_shapefile['lsup'] = adc_shapefile['sup_sem'].apply(np.log)


In [ ]:
fig, ax = plt.subplots(1,1, figsize=(14, 7))

# municipio_shapefile.plot(ax=ax, edgecolor='white',linewidth=3)
if len(adc_shapefile[adc_shapefile['lsup'].isnull()]): adc_shapefile[adc_shapefile['lsup'].isnull()].plot(color='silver', ax=ax)  # empty subsets crash modern geopandas # this shows where nan's are
non_null_df = adc_shapefile[adc_shapefile['lsup'].notnull()]
if len(non_null_df[non_null_df['lsup'] == 0]): non_null_df[non_null_df['lsup'] == 0].plot(color='lightgrey', ax=ax)  # empty subsets crash modern geopandas
non_null_df[non_null_df['lsup'] != 0].plot('lsup', ax=ax,legend=True, vmin=0, vmax=7)

# st_line_shp.plot(ax=ax, edgecolor='dimgrey',linewidth=1)

ax.axis('off')
plt.gca().axes.get_yaxis().set_visible(False)
plt.gca().axes.get_xaxis().set_visible(False)
ax.set_axis_off()

ax.set_title('(Log) hectares harvested of avocado at área de control level in Mexico')
plt.savefig(plot_dir+'avosup_allmx_adc_with_legend.png',bbox_inches='tight',dpi=600, transparent=True)
plt.show()

In [ ]:
# yield_rc_pred existed only interactively; the corrected CA07 pred is
# corr_yield_pred_sup_sem (these error maps are superseded by 3_plot_yields_ca22_maps.py)
if 'yield_rc_pred' not in adc_shapefile.columns:
    adc_shapefile['yield_rc_pred'] = adc_shapefile.get('corr_yield_pred_sup_sem')
adc_shapefile['prederror_rcpred']   = adc_shapefile['yield'] - adc_shapefile['yield_rc_pred']
adc_shapefile['prederror_ls']       = adc_shapefile['yield'] - adc_shapefile['yield_pred_ls']
adc_shapefile['prederror_munyield'] = adc_shapefile['yield'] - adc_shapefile['munyield']


In [ ]:
predvar = 'prederror_ls'
fig, ax = plt.subplots(1,1, figsize=(14, 7))

# municipio_shapefile.plot(ax=ax, edgecolor='white',linewidth=3)
if len(adc_shapefile[adc_shapefile[predvar].isnull()]): adc_shapefile[adc_shapefile[predvar].isnull()].plot(color='silver', ax=ax)  # empty subsets crash modern geopandas # this shows where nan's are
non_null_df = adc_shapefile[adc_shapefile[predvar].notnull()]
non_null_df.plot(predvar, cmap='RdBu', ax=ax,legend=False, vmin=-10, vmax=10)

# st_line_shp.plot(ax=ax, edgecolor='dimgrey',linewidth=0.5)

ax.axis('off')
plt.gca().axes.get_yaxis().set_visible(False)
plt.gca().axes.get_xaxis().set_visible(False)
ax.set_axis_off()

# ax.set_title('Maize yield (tons/hectare harvested) at área de control level in Mexico')
plt.savefig(plot_dir+'maizeyield_adc_pred_error_'+predvar+'_noleg.png',bbox_inches='tight',dpi=600, transparent=True)

plt.show()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(14, 7))

# municipio_shapefile.plot(ax=ax, edgecolor='white',linewidth=3)
if len(adc_shapefile[adc_shapefile['yield'].isnull()]): adc_shapefile[adc_shapefile['yield'].isnull()].plot(color='silver', ax=ax)  # empty subsets crash modern geopandas # this shows where nan's are
non_null_df = adc_shapefile[adc_shapefile['yield'].notnull()]
if len(non_null_df[non_null_df['yield'] == 0]): non_null_df[non_null_df['yield'] == 0].plot(color='lightgrey', ax=ax)  # empty subsets crash modern geopandas
non_null_df[non_null_df['yield'] != 0].plot('yield', ax=ax,legend=True, vmin=0, vmax=12)

# st_line_shp.plot(ax=ax, edgecolor='dimgrey',linewidth=0.5)

ax.axis('off')
plt.gca().axes.get_yaxis().set_visible(False)
plt.gca().axes.get_xaxis().set_visible(False)
ax.set_axis_off()

# ax.set_title('Maize yield (tons/hectare harvested) at área de control level in Mexico')
# plt.savefig(plot_dir+'maizeyield_allmx_adc.png',bbox_inches='tight',dpi=600, transparent=True)
plt.savefig(plot_dir+'maizeyield_allmx_adc_with_legend.png',bbox_inches='tight',dpi=600, transparent=True)

plt.show()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(14, 7))

# municipio_shapefile.plot(ax=ax, edgecolor='white',linewidth=3)
if len(adc_shapefile[adc_shapefile['yield_pred_ls'].isnull()]): adc_shapefile[adc_shapefile['yield_pred_ls'].isnull()].plot(color='silver', ax=ax)  # empty subsets crash modern geopandas # this shows where nan's are
non_null_df = adc_shapefile[adc_shapefile['yield_pred_ls'].notnull()]
if len(non_null_df[non_null_df['yield_pred_ls'] == 0]): non_null_df[non_null_df['yield_pred_ls'] == 0].plot(color='lightgrey', ax=ax)  # empty subsets crash modern geopandas
non_null_df[non_null_df['yield_pred_ls'] != 0].plot('yield_pred_ls', ax=ax, legend=False, vmin=0, vmax=7)

# st_line_shp.plot(ax=ax, edgecolor='dimgrey',linewidth=0.5)

ax.axis('off')
plt.gca().axes.get_yaxis().set_visible(False)
plt.gca().axes.get_xaxis().set_visible(False)
ax.set_axis_off()

# ax.set_title('Maize yield (tons/hectare harvested) at área de control level in Mexico')
plt.savefig(plot_dir+'maizeyield_adc_preds.png',bbox_inches='tight',dpi=600, transparent=True)
plt.show()

In [ ]:
siap_df['muncode'] = siap_df['muncode'].astype(int).astype(str)

municipio_shapefile = municipio_shapefile.merge(siap_df, on='muncode', how='left')


In [ ]:
mun_pred_df['muncode'] = mun_pred_df['muncode'].astype(int).astype(str)
municipio_shapefile = municipio_shapefile.merge(mun_pred_df, on='muncode', how='left')

municipio_shapefile.columns


In [ ]:
municipio_shapefile.to_crs(4326, inplace=True)

In [ ]:
# pred_x arose from an interactive merge-suffix collision; guard (superseded by _2022 map)
if 'pred_x' not in municipio_shapefile.columns:
    print('pred_x not present -- skipping superseded 2018 municipal pred map')
else:
    
    fig, ax = plt.subplots(1,1, figsize=(14, 7))
    
        
    if len(municipio_shapefile[municipio_shapefile['pred_x'].isnull()]): municipio_shapefile[municipio_shapefile['pred_x'].isnull()].plot(color='silver', ax=ax)  # empty subsets crash modern geopandas # this shows where nan's are
    non_null_df = municipio_shapefile[municipio_shapefile['pred_x'].notnull()]
    if len(non_null_df[non_null_df['pred_x'] == 0]): non_null_df[non_null_df['pred_x'] == 0].plot(color='lightgrey', ax=ax)  # empty subsets crash modern geopandas
    non_null_df[non_null_df['pred_x'] != 0].plot('pred_x', ax=ax,legend=False, vmin=0, vmax=12)
    
    st_line_shp.to_crs({'init': 'epsg:4326'}).plot(ax=ax, edgecolor='dimgrey',linewidth=0.75)
    
    ax.axis('off')
    plt.gca().axes.get_yaxis().set_visible(False)
    plt.gca().axes.get_xaxis().set_visible(False)
    ax.set_axis_off()
    
    plt.savefig(plot_dir+'maizeyield_mun_pred_allmx_nolegend_2018.png',bbox_inches='tight',dpi=600, transparent = True)
    plt.show()

In [ ]:

fig, ax = plt.subplots(1,1, figsize=(14, 7))

    
if len(municipio_shapefile[municipio_shapefile['munyield'].isnull()]): municipio_shapefile[municipio_shapefile['munyield'].isnull()].plot(color='silver', ax=ax)  # empty subsets crash modern geopandas # this shows where nan's are
non_null_df = municipio_shapefile[municipio_shapefile['munyield'].notnull()]
if len(non_null_df[non_null_df['munyield'] == 0]): non_null_df[non_null_df['munyield'] == 0].plot(color='lightgrey', ax=ax)  # empty subsets crash modern geopandas
non_null_df[non_null_df['munyield'] != 0].plot('munyield', ax=ax,legend=True, vmin=0, vmax=12)

st_line_shp.to_crs({'init': 'epsg:4326'}).plot(ax=ax, edgecolor='dimgrey',linewidth=0.75)

ax.axis('off')
plt.gca().axes.get_yaxis().set_visible(False)
plt.gca().axes.get_xaxis().set_visible(False)
ax.set_axis_off()
# plt.xlim(3050000,3100000)
# plt.ylim(710000,745000)

# ax.set_title('Maize yield (tons/hectare harvested) at municipality level in Oaxaca state')
plt.savefig(plot_dir+'maizeyield_mun_siap_allmx.png',bbox_inches='tight',dpi=600, transparent = True)
plt.show()

In [ ]:
### Municipal CA2007 census maize yield for the mun-level maps.
### Restored 2026-08-28: this frame was previously defined only interactively
### (the recipe existed as a comment), which broke top-to-bottom execution.
mun_yield_df = adc_df.dropna(subset=['muncode']).copy()
mun_yield_df['muncode'] = pd.to_numeric(mun_yield_df['muncode'], errors='coerce')
mun_yield_df = mun_yield_df.dropna(subset=['muncode'])
mun_yield_df = mun_yield_df.groupby('muncode')[['sup_sem','Q']].sum().reset_index()
mun_yield_df['mun_yield'] = mun_yield_df['Q']/mun_yield_df['sup_sem']
mun_yield_df = mun_yield_df.drop(['sup_sem','Q'], axis=1)
mun_yield_df['muncode'] = mun_yield_df['muncode'].astype(int).astype(str)


In [ ]:
municipio_shapefile = municipio_shapefile.merge(mun_yield_df, on='muncode', how='left')
# The mun-level map cells below use PROJECTED coords (matching st_line_shp
# and the historical panels); cell 38's to_crs(4326) made the frame invisible
# against the projected state lines. Keep a projected copy for plotting.
municipio_proj = municipio_shapefile.to_crs(st_line_shp.crs)


In [ ]:
fig, ax = plt.subplots(1,1, figsize=(14, 7))

    
if len(municipio_proj[municipio_proj.mun_yield.isnull()]): municipio_proj[municipio_proj.mun_yield.isnull()].plot(color='silver', ax=ax)  # empty subsets crash modern geopandas # this shows where nan's are
non_null_df = municipio_proj[municipio_proj.mun_yield.notnull()]
if len(non_null_df[non_null_df.mun_yield == 0]): non_null_df[non_null_df.mun_yield == 0].plot(color='lightgrey', ax=ax)  # empty subsets crash modern geopandas
non_null_df[non_null_df.mun_yield != 0].plot('mun_yield', ax=ax,legend=True, vmin=0, vmax=14)

st_line_shp.plot(ax=ax, edgecolor='dimgrey',linewidth=0.75)

ax.axis('off')
plt.gca().axes.get_yaxis().set_visible(False)
plt.gca().axes.get_xaxis().set_visible(False)
ax.set_axis_off()
# plt.xlim(3050000,3100000)
# plt.ylim(710000,745000)

# ax.set_title('Maize yield (tons/hectare harvested) at municipality level in Oaxaca state')
plt.savefig(plot_dir+'maizeyield_mun_allmx.png',bbox_inches='tight',dpi=600, transparent = True)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(14, 7))

    
if len(municipio_proj[municipio_proj.mun_yield.isnull()]): municipio_proj[municipio_proj.mun_yield.isnull()].plot(color='silver', ax=ax)  # empty subsets crash modern geopandas # this shows where nan's are
non_null_df = municipio_proj[municipio_proj.mun_yield.notnull()]
if len(non_null_df[non_null_df.mun_yield == 0]): non_null_df[non_null_df.mun_yield == 0].plot(color='lightgrey', ax=ax)  # empty subsets crash modern geopandas
non_null_df[non_null_df.mun_yield != 0].plot('mun_yield', ax=ax,legend=True, vmin=0, vmax=5)

st_line_shp.plot(ax=ax, edgecolor='dimgrey',linewidth=1)

ax.axis('off')
plt.gca().axes.get_yaxis().set_visible(False)
plt.gca().axes.get_xaxis().set_visible(False)
ax.set_axis_off()
plt.xlim(3050000,3100000)
plt.ylim(710000,745000)

ax.set_title('Maize yield (tons/hectare harvested) at municipality level in Oaxaca state')
plt.savefig(plot_dir+'maizeyield_mun.png',bbox_inches='tight',dpi=600)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(14, 7))

municipio_proj.plot(ax=ax, facecolor='none', edgecolor='white', linewidth=3)  # background invisible, as published
if len(adc_shapefile[adc_shapefile['yield'].isnull()]): adc_shapefile[adc_shapefile['yield'].isnull()].plot(color='silver', ax=ax)  # empty subsets crash modern geopandas # this shows where nan's are
non_null_df = adc_shapefile[adc_shapefile['yield'].notnull()]
if len(non_null_df[non_null_df['yield'] == 0]): non_null_df[non_null_df['yield'] == 0].plot(color='lightgrey', ax=ax)  # empty subsets crash modern geopandas
non_null_df[non_null_df['yield'] != 0].plot('yield', ax=ax,legend=True, vmin=0, vmax=5)

st_line_shp.plot(ax=ax, edgecolor='dimgrey',linewidth=1)

ax.axis('off')
plt.gca().axes.get_yaxis().set_visible(False)
plt.gca().axes.get_xaxis().set_visible(False)
ax.set_axis_off()
plt.xlim(3050000,3100000)
plt.ylim(710000,745000)

ax.set_title('Maize yield (tons/hectare harvested) at área de control level in Oaxaca state')
plt.savefig(plot_dir+'maizeyield_adc.png',bbox_inches='tight',dpi=600)
plt.show()